# N3: Sample Stata Analysis

<a href="https://colab.research.google.com/github/cmg777/claude4data/blob/master/notebooks/notebook-03.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## Overview

This notebook demonstrates the Stata workflow: generating synthetic data, producing labeled figures and tables that are embedded in the manuscript via Quarto’s `{{< embed >}}` shortcode. It mirrors `notebook-01` (Python) using Stata.

**Prerequisites:** Stata must be installed and the `nbstata` Jupyter kernel must be registered. See `notebooks/README.md` for setup instructions.

In [1]:
* Reproducibility setup
clear all
set seed 42

In [2]:
* Generate synthetic cross-sectional data (80 regions, 4 clusters)
quietly {
    set obs 80
    gen region_id = _n
    gen region = "North" if region_id <= 20
    replace region = "South" if region_id > 20 & region_id <= 40
    replace region = "East"  if region_id > 40 & region_id <= 60
    replace region = "West"  if region_id > 60

    gen log_gdp = rnormal(9.5, 0.6)
    gen gdp_per_capita = exp(log_gdp)
    gen life_expectancy = 50 + 15 * (1 - exp(-gdp_per_capita / 20000)) + rnormal(0, 2)
}
describe


Contains data
 Observations:            80                  
    Variables:             5                  
-------------------------------------------------------------------------------
Variable      Storage   Display    Value
    name         type    format    label      Variable label
-------------------------------------------------------------------------------
region_id       float   %9.0g                 
region          str5    %9s                   
log_gdp         float   %9.0g                 
gdp_per_capita  float   %9.0g                 
life_expectancy float   %9.0g                 
-------------------------------------------------------------------------------
Sorted by: 
     Note: Dataset has changed since last saved.

In [3]:
twoway ///
    (scatter life_expectancy gdp_per_capita if region == "North", mcolor(blue)    msymbol(circle)) ///
    (scatter life_expectancy gdp_per_capita if region == "South", mcolor(red)     msymbol(diamond)) ///
    (scatter life_expectancy gdp_per_capita if region == "East",  mcolor(green)   msymbol(triangle)) ///
    (scatter life_expectancy gdp_per_capita if region == "West",  mcolor(orange)  msymbol(square)), ///
    title("GDP per Capita vs. Life Expectancy") ///
    xtitle("GDP per capita (USD)") ///
    ytitle("Life expectancy (years)") ///
    legend(order(1 "North" 2 "South" 3 "East" 4 "West") title("Region"))

In [4]:
tabstat gdp_per_capita life_expectancy, by(region) stat(mean sd count) format(%9.1f)


Summary statistics: Mean, SD, N
Group variable: region 

region |  gdp_pe~a  life_e~y
-------+--------------------
  East |   18540.3      58.2
       |   12777.3       3.7
       |      20.0      20.0
-------+--------------------
 North |   13941.3      57.3
       |    6845.2       2.8
       |      20.0      20.0
-------+--------------------
 South |   15757.6      57.4
       |    8297.8       3.1
       |      20.0      20.0
-------+--------------------
  West |   21923.5      59.7
       |   12889.5       4.3
       |      20.0      20.0
-------+--------------------
 Total |   17540.7      58.2
       |   10782.0       3.6
       |      80.0      80.0
----------------------------